[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/086.%20The%20Capacitated%20Facility%20Location%20Problem%20%28CFLP%29/P86-Tier-2_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 86. The Capacitated Facility Location Problem (CFLP)

## Tier 2: The Classic Heuristic (Beam Search)

### Key Assumptions
- Customer demand is deterministic (single scenario for heuristic simplicity)
- Fixed costs and transportation costs are known and constant
- Facility capacities are hard constraints that cannot be exceeded
- All customer demand must be satisfied by open facilities
- Search space is explored using beam width parameter β

### Approach (Step-by-Step)

Beam Search is a heuristic algorithm that explores the search space by maintaining a "beam" of the most promising partial solutions at each level. It's a compromise between:
- **Greedy search** (keeps only 1 best solution)
- **Breadth-first search** (keeps all solutions)

**Algorithm Structure:**
1. **Initialization**: Start with empty set of open facilities
2. **Level-by-level expansion**: Consider each potential facility sequentially
3. **Branching**: For each partial solution, create two children (open/not open facility)
4. **Evaluation**: Calculate total cost for each child solution
5. **Pruning**: Keep only the top β solutions (beam width)
6. **Termination**: Return best solution from final beam

### What to Look for in the Results
- How beam width affects solution quality and computation time
- Comparison with optimal solution from Tier 1
- Trade-off between exploration (larger β) and efficiency (smaller β)
- Pattern of facility openings as search progresses

### Concrete Example

We'll apply Beam Search to a larger instance:
- **8 potential facilities** (more complex than Tier 1)
- **6 customers** with deterministic demand
- **Beam widths**: β=2, 3, 5 for comparison

This demonstrates how Beam Search handles larger problems where exact optimization becomes difficult.

In [1]:
from typing import Tuple, List, Dict, Set

# Import required packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Dict, Tuple, Set
import heapq
import time
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class Facility:
    """Represents a potential facility location"""
    id: int
    name: str
    fixed_cost: float
    capacity: float
    x_coord: float
    y_coord: float

@dataclass
class Customer:
    """Represents a customer location"""
    id: int
    name: str
    demand: float
    x_coord: float
    y_coord: float

@dataclass
class BeamNode:
    """Represents a node in the beam search tree"""
    open_facilities: Set[int]
    cost: float
    level: int  # Which facility we're considering
    
    def __lt__(self, other):
        # For heapq (min-heap) - lower cost is better
        return self.cost < other.cost

In [ ]:
# Define larger instance for Beam Search demonstration

# 8 potential facilities with varying characteristics
facilities = [
    Facility(1, "North Hub", 120000, 1000, 20, 80),   # Large capacity, high cost
    Facility(2, "Central DC", 90000, 800, 50, 50),     # Medium capacity, medium cost
    Facility(3, "South DC", 70000, 600, 80, 20),     # Small capacity, low cost
    Facility(4, "East Facility", 85000, 700, 90, 60), # Medium capacity, medium-high cost
    Facility(5, "West Facility", 75000, 650, 10, 40), # Medium-small capacity, medium cost
    Facility(6, "Northeast DC", 95000, 750, 70, 85),  # Medium capacity, high cost
    Facility(7, "Southwest DC", 65000, 550, 30, 15),  # Small capacity, low cost
    Facility(8, "Central Hub", 110000, 900, 55, 55)   # Large capacity, high cost
]

# 6 customers with deterministic demand
customers = [
    Customer(1, "Store A", 300, 25, 75),
    Customer(2, "Store B", 450, 45, 65),
    Customer(3, "Store C", 280, 75, 35),
    Customer(4, "Store D", 380, 85, 55),
    Customer(5, "Store E", 320, 35, 25),
    Customer(6, "Store F", 410, 65, 85)
]

print("Beam Search Instance Summary:")
print(f"Facilities: {len(facilities)}")
print(f"Customers: {len(customers)}")
print(f"Total Demand: {sum(c.demand for c in customers)}")
print(f"Total Capacity: {sum(f.capacity for f in facilities)}")

Beam Search Instance Summary:
Facilities: 8
Customers: 6
Total Demand: 2140
Total Capacity: 5950


In [ ]:
# Calculate transportation costs
def calculate_transport_cost(facility: Facility, customer: Customer) -> float:
    """Calculate transportation cost based on Euclidean distance"""
    distance = np.sqrt((facility.x_coord - customer.x_coord)**2 + 
                      (facility.y_coord - customer.y_coord)**2)
    cost_per_unit = 8.0  # $/unit/distance
    return distance * cost_per_unit

# Build transportation cost matrix
transport_costs = {}
for facility in facilities:
    for customer in customers:
        transport_costs[(facility.id, customer.id)] = calculate_transport_cost(facility, customer)

# Display cost matrix
cost_matrix = pd.DataFrame(
    [[transport_costs[(f.id, c.id)] for c in customers] for f in facilities],
    index=[f"{f.name}" for f in facilities],
    columns=[f"{c.name}" for c in customers]
)

print("Transportation Cost Matrix ($/unit):")
print(cost_matrix.round(2))

Transportation Cost Matrix ($/unit):
               Store A  Store B  Store C  Store D  Store E  Store F
North Hub        56.57   233.24   568.51   557.14   456.07   362.22
Central DC      282.84   126.49   233.24   282.84   233.24   304.63
South DC        622.25   456.07   126.49   282.84   362.22   533.67
East Facility   533.67   362.22   233.24    56.57   521.54   282.84
West Facility   304.63   344.09   521.54   611.88   233.24   568.51
Northeast DC    368.78   256.12   402.00   268.33   555.70    40.00
Southwest DC    481.66   417.61   393.95   544.06    89.44   626.10
Central Hub     288.44   113.14   226.27   240.00   288.44   252.98


In [ ]:
def evaluate_solution(open_facilities: Set[int], 
                      facilities: List[Facility],
                      customers: List[Customer],
                      transport_costs: Dict[Tuple[int, int], float]) -> float:
    """
    Evaluate the total cost of a partial or complete solution
    Uses greedy assignment for customer allocation
    
    Returns:
        Total cost (fixed + transport) or infinity if infeasible
    """
    
    # Calculate fixed costs
    fixed_cost = sum(f.fixed_cost for f in facilities if f.id in open_facilities)
    
    # If no facilities are open, return infinity (infeasible)
    if not open_facilities:
        return float('inf')
    
    # Greedy customer assignment
    transport_cost = 0
    facility_loads = {f.id: 0 for f in facilities if f.id in open_facilities}
    
    # Sort customers by demand (largest first) for better assignment
    sorted_customers = sorted(customers, key=lambda c: c.demand, reverse=True)
    
    for customer in sorted_customers:
        # Find cheapest open facility with available capacity
        best_facility = None
        best_cost = float('inf')
        
        for facility_id in open_facilities:
            facility = next(f for f in facilities if f.id == facility_id)
            
            # Check capacity constraint
            if facility_loads[facility_id] + customer.demand <= facility.capacity:
                cost = transport_costs[(facility_id, customer.id)]
                if cost < best_cost:
                    best_cost = cost
                    best_facility = facility_id
        
        # If no facility can serve this customer, solution is infeasible
        if best_facility is None:
            return float('inf')
        
        # Assign customer to best facility
        transport_cost += best_cost * customer.demand
        facility_loads[best_facility] += customer.demand
    
    return fixed_cost + transport_cost

In [ ]:
def beam_search_cflp(facilities: List[Facility],
                    customers: List[Customer],
                    transport_costs: Dict[Tuple[int, int], float],
                    beam_width: int = 3) -> Dict:
    """
    Beam Search algorithm for CFLP
    
    Args:
        facilities: List of potential facilities
        customers: List of customers with demand
        transport_costs: Dictionary of transportation costs
        beam_width: Number of solutions to keep at each level (β)
    
    Returns:
        Dictionary containing solution and search statistics
    """
    
    start_time = time.time()
    search_progress = []
    
    # Initialize beam with root node (no facilities open)
    root_node = BeamNode(set(), 0, 0)
    beam = [root_node]
    
    # Process each facility level by level
    for level, facility in enumerate(facilities):
        candidates = []
        
        # Generate children for each node in current beam
        while beam:
            current_node = beam.pop()
            
            # Child 1: Do NOT open this facility
            child1_open_facilities = current_node.open_facilities.copy()
            child1_cost = evaluate_solution(
                child1_open_facilities, facilities, customers, transport_costs
            )
            child1 = BeamNode(child1_open_facilities, child1_cost, level + 1)
            candidates.append(child1)
            
            # Child 2: Open this facility
            child2_open_facilities = current_node.open_facilities.copy()
            child2_open_facilities.add(facility.id)
            child2_cost = evaluate_solution(
                child2_open_facilities, facilities, customers, transport_costs
            )
            child2 = BeamNode(child2_open_facilities, child2_cost, level + 1)
            candidates.append(child2)
        
        # Sort candidates by cost and keep top beam_width
        candidates.sort(key=lambda x: x.cost)
        beam = candidates[:beam_width]
        
        # Record search progress
        best_cost = beam[0].cost if beam else float('inf')
        search_progress.append({
            'level': level + 1,
            'facility': facility.name,
            'best_cost': best_cost,
            'beam_size': len(beam)
        })
        
        print(f"Level {level + 1}/{len(facilities)}: {facility.name}")
        print(f"  Best cost so far: ${best_cost:,.2f}")
        print(f"  Beam size: {len(beam)}")
    
    # Return best solution from final beam
    best_node = beam[0] if beam else None
    end_time = time.time()
    
    return {
        'solution': best_node,
        'search_time': end_time - start_time,
        'search_progress': search_progress,
        'beam_width': beam_width,
        'total_nodes_explored': len(search_progress) * beam_width * 2  # Approximate
    }

In [ ]:
# Run Beam Search with different beam widths
beam_widths = [2, 3, 5]
results = {}

for beta in beam_widths:
    print(f"\n{'='*60}")
    print(f"BEAM SEARCH WITH WIDTH β = {beta}")
    print(f"{'='*60}")
    
    result = beam_search_cflp(facilities, customers, transport_costs, beta)
    results[beta] = result
    
    print(f"\nFinal Results for β = {beta}:")
    print(f"Search Time: {result['search_time']:.3f} seconds")
    print(f"Total Cost: ${result['solution'].cost:,.2f}")
    print(f"Open Facilities: {sorted(result['solution'].open_facilities)}")
    
    # Show facility names
    open_facility_names = [
        f.name for f in facilities if f.id in result['solution'].open_facilities
    ]
    print(f"Facility Names: {open_facility_names}")


BEAM SEARCH WITH WIDTH β = 2
Level 1/8: North Hub
  Best cost so far: $inf
  Beam size: 2
Level 2/8: Central DC
  Best cost so far: $inf
  Beam size: 2
Level 3/8: South DC
  Best cost so far: $843,698.20
  Beam size: 2
Level 4/8: East Facility
  Best cost so far: $802,280.15
  Beam size: 2
Level 5/8: West Facility
  Best cost so far: $802,280.15
  Beam size: 2
Level 6/8: Northeast DC
  Best cost so far: $641,730.45
  Beam size: 2
Level 7/8: Southwest DC
  Best cost so far: $641,730.45
  Beam size: 2
Level 8/8: Central Hub
  Best cost so far: $641,730.45
  Beam size: 2

Final Results for β = 2:
Search Time: 0.001 seconds
Total Cost: $641,730.45
Open Facilities: [1, 2, 4, 6]
Facility Names: ['North Hub', 'Central DC', 'East Facility', 'Northeast DC']

BEAM SEARCH WITH WIDTH β = 3
Level 1/8: North Hub
  Best cost so far: $inf
  Beam size: 2
Level 2/8: Central DC
  Best cost so far: $inf
  Beam size: 3
Level 3/8: South DC
  Best cost so far: $843,698.20
  Beam size: 3
Level 4/8: East Faci

In [ ]:
# Compare results across different beam widths
print("\n" + "="*80)
print("COMPARISON ACROSS BEAM WIDTHS")
print("="*80)

comparison_data = []
for beta, result in results.items():
    comparison_data.append({
        'Beam Width': beta,
        'Total Cost': result['solution'].cost,
        'Search Time (s)': result['search_time'],
        'Open Facilities': len(result['solution'].open_facilities),
        'Nodes Explored': result['total_nodes_explored']
    })

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.round(2))

# Find best solution
best_beta = min(results.keys(), key=lambda b: results[b]['solution'].cost)
best_result = results[best_beta]

print(f"\nBest Solution: β = {best_beta}")
print(f"Cost: ${best_result['solution'].cost:,.2f}")
print(f"Open Facilities: {sorted(best_result['solution'].open_facilities)}")


COMPARISON ACROSS BEAM WIDTHS
   Beam Width  Total Cost  Search Time (s)  Open Facilities  Nodes Explored
0           2   641730.45              0.0                4              32
1           3   681841.30              0.0                5              48
2           5   681841.30              0.0                5              80

Best Solution: β = 2
Cost: $641,730.45
Open Facilities: [1, 2, 4, 6]


In [ ]:
# Detailed analysis of best solution
best_solution = best_result['solution']
open_facilities = [f for f in facilities if f.id in best_solution.open_facilities]

print("\n" + "="*60)
print("DETAILED ANALYSIS OF BEST SOLUTION")
print("="*60)

# Calculate cost breakdown
fixed_costs = sum(f.fixed_cost for f in open_facilities)
print(f"\nOpen Facilities ({len(open_facilities)}):")
for facility in open_facilities:
    print(f"  {facility.name}: Fixed cost ${facility.fixed_cost:,}, Capacity {facility.capacity}")

print(f"\nTotal Fixed Costs: ${fixed_costs:,}")
print(f"Transportation Costs: ${best_solution.cost - fixed_costs:,}")

# Greedy assignment for detailed analysis
facility_loads = {f.id: 0 for f in open_facilities}
assignments = {}

sorted_customers = sorted(customers, key=lambda c: c.demand, reverse=True)

print(f"\nCustomer Assignments:")
for customer in sorted_customers:
    # Find assignment (same logic as evaluate_solution)
    best_facility_id = None
    best_cost = float('inf')
    
    for facility in open_facilities:
        if facility_loads[facility.id] + customer.demand <= facility.capacity:
            cost = transport_costs[(facility.id, customer.id)]
            if cost < best_cost:
                best_cost = cost
                best_facility_id = facility.id
    
    facility_name = next(f.name for f in open_facilities if f.id == best_facility_id)
    assignments[customer.id] = facility_name
    facility_loads[best_facility_id] += customer.demand
    
    print(f"  {customer.name} (demand {customer.demand}) → {facility_name} (cost ${best_cost * customer.demand:.2f})")

print(f"\nFacility Utilization:")
for facility in open_facilities:
    utilization = (facility_loads[facility.id] / facility.capacity) * 100
    print(f"  {facility.name}: {facility_loads[facility.id]}/{facility.capacity} ({utilization:.1f}%)")


DETAILED ANALYSIS OF BEST SOLUTION

Open Facilities (4):
  North Hub: Fixed cost $120,000, Capacity 1000
  Central DC: Fixed cost $90,000, Capacity 800
  East Facility: Fixed cost $85,000, Capacity 700
  Northeast DC: Fixed cost $95,000, Capacity 750

Total Fixed Costs: $390,000
Transportation Costs: $251,730.45225586626

Customer Assignments:
  Store B (demand 450) → Central DC (cost $56921.00)
  Store F (demand 410) → Northeast DC (cost $16400.00)
  Store D (demand 380) → East Facility (cost $21496.05)
  Store E (demand 320) → Central DC (cost $74636.18)
  Store A (demand 300) → North Hub (cost $16970.56)
  Store C (demand 280) → East Facility (cost $65306.66)

Facility Utilization:
  North Hub: 300/1000 (30.0%)
  Central DC: 770/800 (96.2%)
  East Facility: 660/700 (94.3%)
  Northeast DC: 410/750 (54.7%)


In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Beam Search CFLP Analysis', fontsize=16, fontweight='bold')

# 1. Cost comparison across beam widths
ax1 = axes[0, 0]
widths = list(results.keys())
costs = [results[b]['solution'].cost for b in widths]
ax1.bar(widths, costs, color='steelblue', alpha=0.7)
ax1.set_xlabel('Beam Width (β)')
ax1.set_ylabel('Total Cost ($)')
ax1.set_title('Solution Quality vs Beam Width')
ax1.grid(True, alpha=0.3)
for i, (w, c) in enumerate(zip(widths, costs)):
    ax1.text(w, c + max(costs)*0.01, f'${c:,.0f}', ha='center', va='bottom')

# 2. Search time comparison
ax2 = axes[0, 1]
times = [results[b]['search_time'] for b in widths]
ax2.bar(widths, times, color='coral', alpha=0.7)
ax2.set_xlabel('Beam Width (β)')
ax2.set_ylabel('Search Time (seconds)')
ax2.set_title('Computational Time vs Beam Width')
ax2.grid(True, alpha=0.3)
for i, (w, t) in enumerate(zip(widths, times)):
    ax2.text(w, t + max(times)*0.01, f'{t:.3f}s', ha='center', va='bottom')

# 3. Search progress (cost reduction)
ax3 = axes[0, 2]
for beta, result in results.items():
    progress = result['search_progress']
    levels = [p['level'] for p in progress]
    costs = [p['best_cost'] if p['best_cost'] != float('inf') else 1e6 for p in progress]
    ax3.plot(levels, costs, marker='o', label=f'β={beta}', linewidth=2, markersize=4)

ax3.set_xlabel('Search Level')
ax3.set_ylabel('Best Cost Found ($)')
ax3.set_title('Search Progress by Level')
ax3.legend()
ax3.grid(True, alpha=0.3)
ax3.set_yscale('log')  # Log scale due to large cost range

# 4. Facility locations and assignments
ax4 = axes[1, 0]
# Plot all facilities
for facility in facilities:
    color = 'red' if facility.id in best_solution.open_facilities else 'lightgray'
    marker = 's' if facility.id in best_solution.open_facilities else 's'
    size = 300 if facility.id in best_solution.open_facilities else 100
    ax4.scatter(facility.x_coord, facility.y_coord, c=color, marker=marker, s=size,
               edgecolors='black', linewidths=2)
    ax4.annotate(f"{facility.name}", (facility.x_coord, facility.y_coord),
                xytext=(5, 5), textcoords='offset points', fontsize=8)

# Plot customers
for customer in customers:
    ax4.scatter(customer.x_coord, customer.y_coord, c='blue', marker='o', s=150,
               edgecolors='black', linewidths=1)
    ax4.annotate(f"{customer.name}\n({customer.demand})", (customer.x_coord, customer.y_coord),
                xytext=(5, 5), textcoords='offset points', fontsize=8)

ax4.set_xlabel('X Coordinate')
ax4.set_ylabel('Y Coordinate')
ax4.set_title('Best Solution: Facility Locations and Assignments')
ax4.grid(True, alpha=0.3)

# 5. Cost breakdown for best solution
ax5 = axes[1, 1]
fixed_cost = sum(f.fixed_cost for f in open_facilities)
transport_cost = best_solution.cost - fixed_cost
costs_breakdown = [fixed_cost, transport_cost]
labels = ['Fixed Costs', 'Transport Costs']
colors = ['#ff7f0e', '#1f77b4']
ax5.pie(costs_breakdown, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax5.set_title(f'Cost Structure (β={best_beta})')

# 6. Facility utilization
ax6 = axes[1, 2]
facility_names = [f.name for f in open_facilities]
utilization_rates = [(facility_loads[f.id] / f.capacity) * 100 for f in open_facilities]
ax6.barh(facility_names, utilization_rates, color='lightgreen', alpha=0.7)
ax6.set_xlabel('Utilization Rate (%)')
ax6.set_title('Facility Utilization in Best Solution')
ax6.grid(True, alpha=0.3)
ax6.axvline(x=80, color='red', linestyle='--', alpha=0.7, label='80% Target')
ax6.legend()

for i, (name, util) in enumerate(zip(facility_names, utilization_rates)):
    ax6.text(util + 1, i, f'{util:.1f}%', va='center')

plt.tight_layout()
plt.show()

In [ ]:
# What-if analysis: Effect of beam width on solution diversity
print("\n" + "="*60)
print("WHAT-IF ANALYSIS: BEAM WIDTH IMPACT")
print("="*60)

# Analyze solution diversity
solution_sets = {}
for beta, result in results.items():
    solution_sets[beta] = set(result['solution'].open_facilities)

print("Solution Diversity Analysis:")
for i, beta1 in enumerate(sorted(solution_sets.keys())):
    for beta2 in sorted(solution_sets.keys())[i+1:]:
        set1 = solution_sets[beta1]
        set2 = solution_sets[beta2]
        intersection = len(set1.intersection(set2))
        union = len(set1.union(set2))
        similarity = intersection / union if union > 0 else 0
        print(f"β={beta1} vs β={beta2}: {intersection} common facilities, {similarity:.1%} similarity")

# Recommendation based on analysis
print("\nRecommendations:")
if best_beta == max(beam_widths):
    print("- Larger beam width (β=5) provides best solution quality")
    print("- Use β=5 when solution quality is more important than speed")
elif best_beta == min(beam_widths):
    print("- Smaller beam width (β=2) provides good balance")
    print("- Use β=2 for fast solutions with acceptable quality")
else:
    print("- Medium beam width (β=3) provides best compromise")
    print("- Use β=3 for balanced performance")

print(f"\nBest compromise solution quality: ${best_solution.cost:,.2f}")
print(f"Search time: {best_result['search_time']:.3f} seconds")
print(f"Nodes explored: ~{best_result['total_nodes_explored']:,}")


WHAT-IF ANALYSIS: BEAM WIDTH IMPACT
Solution Diversity Analysis:
β=2 vs β=3: 4 common facilities, 80.0% similarity
β=2 vs β=5: 4 common facilities, 80.0% similarity
β=3 vs β=5: 5 common facilities, 100.0% similarity

Recommendations:
- Smaller beam width (β=2) provides good balance
- Use β=2 for fast solutions with acceptable quality

Best compromise solution quality: $641,730.45
Search time: 0.001 seconds
Nodes explored: ~32


### Why This Tier Exists vs Tier 1

**Tier 2 (Beam Search) addresses key limitations of Tier 1:**

**Scalability Advantages:**
- **Combinatorial explosion**: Tier 1 MIP struggles with >20 facilities due to 2^n combinations
- **Beam Search**: Controls search space using beam width β, making larger problems tractable
- **Linear complexity**: O(n × β) vs exponential for exact methods

**Practical Benefits:**
- **Faster solution times**: Seconds vs minutes/hours for large instances
- **Memory efficiency**: Only stores β solutions vs full search tree
- **Anytime algorithm**: Can stop early and return best solution found so far

**Quality vs Speed Trade-off:**
- **Adjustable beam width**: β controls exploration vs exploitation balance
- **Empirical performance**: Often finds near-optimal solutions with β=3-5
- **Deterministic**: Same result every run (unlike some metaheuristics)

### Pros and Cons

**Advantages:**
✅ **Scalable** - Handles much larger problems than exact methods
✅ **Fast** - Solution times in seconds for moderate instances
✅ **Controllable quality** - Beam width parameter tunes solution quality
✅ **Deterministic** - Reproducible results
✅ **Memory efficient** - Linear memory usage

**Disadvantages:**
❌ **No optimality guarantee** - May miss optimal solution
❌ **Parameter sensitive** - Performance depends on beam width choice
❌ **Greedy assignment** - Uses heuristic for customer allocation
❌ **Limited exploration** - May get stuck in local optima

### When to Use This Tier

**Ideal for:**
- **Large-scale problems** (20-100 facilities) where exact methods fail
- **Time-critical decisions** requiring fast solutions
- **Preliminary analysis** before detailed optimization
- **What-if scenario analysis** with multiple configurations
- **Resource-constrained environments** with limited computing power

**Not ideal for:**
- **Small problems** where exact methods are feasible
- **Regulatory requirements** demanding proven optimality
- **High-stakes strategic decisions** where optimality is critical
- **Problems with complex constraints** beyond basic capacity limits